In [1]:
import pandas as pd
import lmoments3 as lm
from lmoments3 import distr
import os

# Station mapping from name to file name (station number)
station_mapping = {
    'JPS. KEMAMAN': '0600011RF',
    'JAMBATAN AIR PUTIH': '0600141RF',
    'KG. LA': '0700131RF',
    'KG. DURA': '0670181RF',
    'SG. GAWI': '0670281RF',
    'PENGKALAN NYIREH': '0700011RF',
    'INST. PERTANIAN BESUT': '0690051RF',
    'KG. BATU HAMPAR': '0680081RF',
    'KG. SELADANG': '0680071RF',
    'KG. SG. TONG': '0670211RF',
    'KG. BAN HO': '0600151RF',
    'HULU JABOR A': '0580041RF',
    'S.M.K. SULTAN OMAR': '0630011RF',
    'JAMBATAN TEBAK': '0600131RF',
    'KG. MENERONG': '0670221RF',
    'KG. EMBONG SEKAYU': '0670251RF',
    'RUMAH PAM PAYA KEMPIAN': '0551621RF',
    'JAMBATAN JERANGAU': '0630121RF',
    'AL MUKTAFI BILLAH SHAH': '0620081RF',
    'SETOR JPS. K. TERENGGANU': '0670051RF'
}

# Reverse the mapping to find station names based on file names
file_to_station = {v: k for k, v in station_mapping.items()}

# Define the directory containing your CSV files
directory = 'C:\\Users\\khfy3\\OneDrive\\Desktop\\Master Final Sem\\Coding\\Data\\Annual'

# Directory for saving results
results_directory = os.path.join(directory, 'Analysis Results')
if not os.path.exists(results_directory):
    os.makedirs(results_directory)

# Prepare a list to hold all results
all_results = []

# List all CSV files in the directory
for filename in os.listdir(directory):
    if filename.endswith(".csv") and filename not in ['estimated_parameters_results.csv']:
        file_path = os.path.join(directory, filename)
        station_number = filename[:-4]  # Remove the .csv extension
        
        # Match the station number to the station name
        station_name = file_to_station.get(station_number, "Unknown Station")
        
        # Load data from CSV file
        data = pd.read_csv(file_path)['Value (mm)']
        
        # Estimate L-moments from the data
        l_moments = lm.lmom_ratios(data, nmom=4)
        
        # Fit distributions and estimate parameters
        distributions = ['gum', 'nor', 'exp', 'gev', 'glo', 'gno', 'gpa', 'pe3', 'kap']
        estimated_parameters = {}
        for dist_name in distributions:
            dist = getattr(distr, dist_name)
            try:
                params = dist.lmom_fit(data)
                formatted_params = {k: round(v, 4) for k, v in params.items()}  # Format params to 4 decimal places
            except ValueError as e:
                formatted_params = "Invalid"  # Handle exceptions for invalid L-moments
            estimated_parameters[dist_name] = formatted_params
        
        # Append results for this file including station name and number
        all_results.append({
            'Station Name': station_name,
            'Station Number': station_number,
            **{f"{dist}_params": str(params) for dist, params in estimated_parameters.items()}
        })

# Convert all results to a DataFrame
results_df = pd.DataFrame(all_results)

# Save results to CSV
results_csv_path = os.path.join(results_directory, 'enhanced_estimated_parameters_results.csv')
results_df.to_csv(results_csv_path, index=False)

print(f"Results have been saved to {results_csv_path}")


Results have been saved to C:\Users\khfy3\OneDrive\Desktop\Master Final Sem\Coding\Data\Annual\Analysis Results\enhanced_estimated_parameters_results.csv
